Gemini thread saver for Chrome Debugger console


In [ ]:
(async function extractAndStreamGemini() {
  const sleep = (ms) => new Promise((resolve) => setTimeout(resolve, ms));

  // Remove existing UI control if script is re-run
  const existingBtn = document.getElementById("gemini-dir-export-btn");
  if (existingBtn) existingBtn.remove();

  // Create UI Button
  const startBtn = document.createElement("button");
  startBtn.id = "gemini-dir-export-btn";
  startBtn.innerText = "Select Root Folder & Export Gemini Threads";

  Object.assign(startBtn.style, {
    position: "fixed",
    bottom: "30px",
    right: "30px",
    zIndex: "999999",
    padding: "14px 20px",
    backgroundColor: "#1a73e8",
    color: "#ffffff",
    border: "none",
    borderRadius: "8px",
    fontSize: "15px",
    fontWeight: "bold",
    cursor: "pointer",
    boxShadow: "0 4px 12px rgba(0,0,0,0.3)"
  });

  document.body.appendChild(startBtn);

  const sanitizeFileName = (name) => {
    return (name || "Untitled_Thread")
      .replace(/[\\/:*?"<>|]/g, "_")
      .replace(/\s+/g, "_")
      .slice(0, 60);
  };

  const getMessageHash = (parsed) => {
    return `${parsed.role}_${parsed.text.slice(0, 120)}_${parsed.images.length}_${parsed.links.length}`;
  };

  const extractMessageData = (node) => {
    const tagName = node.tagName.toLowerCase();
    const isUser = tagName.includes("user") || node.classList.contains("user-query");
    const role = isUser ? "user" : "model";
    
    const text = node.innerText ? node.innerText.trim() : "";
    const images = Array.from(node.querySelectorAll("img"))
      .map((img) => img.src || img.getAttribute("data-src"))
      .filter((src) => src && !src.startsWith("data:image/svg"));

    const links = Array.from(node.querySelectorAll("a"))
      .map((link) => ({ text: link.innerText.trim(), url: link.href }))
      .filter((item) => item.url && !item.url.startsWith("javascript:"));

    return { role, text, images, links, extractedAt: new Date().toISOString() };
  };

  // Improved container locator for Gemini's layout
  const findScrollContainer = () => {
    const selectors = [
      ".conversation-container",
      "chat-app",
      "main",
      "[class*='scroll']",
      "[class*='conversation']"
    ];

    for (const sel of selectors) {
      const els = Array.from(document.querySelectorAll(sel));
      const match = els.find((el) => {
        const style = window.getComputedStyle(el);
        const overflow = style.overflowY || style.overflow;
        return (overflow === "auto" || overflow === "scroll") && el.scrollHeight > el.clientHeight;
      });
      if (match) return match;
    }
    return document.scrollingElement || document.documentElement || document.body;
  };

  // Robust multi-strategy scroll to top
  const forceScrollToTop = (container) => {
    // Strategy 1: Find the first message element in DOM and force scroll it into view
    const firstMsg = document.querySelector("user-query, model-response, .user-query, .model-response");
    if (firstMsg) {
      firstMsg.scrollIntoView({ behavior: "instant", block: "start" });
    }

    // Strategy 2: Programmatically set container scroll position
    if (container && container !== document.body && container !== document.documentElement) {
      container.scrollTop = 0;
      container.scrollTo?.({ top: 0, behavior: "instant" });
      container.dispatchEvent(new Event("scroll", { bubbles: true }));
    }

    // Strategy 3: Window / document scroll
    window.scrollTo({ top: 0, behavior: "instant" });
    document.documentElement.scrollTop = 0;
    document.body.scrollTop = 0;
    window.dispatchEvent(new Event("scroll", { bubbles: true }));
  };

  const syncExistingHashes = async (threadDirHandle) => {
    const existingHashes = new Set();
    let maxBatchIndex = 0;

    try {
      for await (const entry of threadDirHandle.values()) {
        if (entry.kind === "file" && entry.name.endsWith(".json") && entry.name.startsWith("batch_")) {
          const match = entry.name.match(/batch_(\d+)_/);
          if (match) {
            const idx = parseInt(match[1], 10);
            if (idx > maxBatchIndex) maxBatchIndex = idx;
          }

          try {
            const file = await entry.getFile();
            const text = await file.text();
            const json = JSON.parse(text);
            if (json.messages && Array.isArray(json.messages)) {
              json.messages.forEach((msg) => existingHashes.add(getMessageHash(msg)));
            }
          } catch (e) {
            console.warn(`Could not parse existing batch ${entry.name}:`, e);
          }
        }
      }
    } catch (err) {
      console.warn("Error reading thread directory:", err);
    }

    return { existingHashes, nextBatchIndex: maxBatchIndex + 1 };
  };

  const processCurrentThread = async (threadDirHandle, threadTitle) => {
    const container = findScrollContainer();
    const { existingHashes, nextBatchIndex } = await syncExistingHashes(threadDirHandle);
    
    let batchIndex = nextBatchIndex;
    let newSavedMessagesCount = 0;
    let unchangedCount = 0;
    let previousNodeCount = 0;

    console.log(`Processing thread: "${threadTitle}". Found ${existingHashes.size} previously saved items.`);

    while (unchangedCount < 4) {
      const messageNodes = Array.from(
        document.querySelectorAll("user-query, model-response, .user-query, .model-response, [data-test-id='user-query']")
      );

      const batchNewMessages = [];

      for (const node of messageNodes) {
        const parsed = extractMessageData(node);
        if (!parsed.text) continue;

        const hash = getMessageHash(parsed);

        if (!existingHashes.has(hash)) {
          existingHashes.add(hash);
          batchNewMessages.push(parsed);
        }
      }

      if (batchNewMessages.length > 0) {
        const now = new Date();
        const timestamp = now.toISOString().replace(/[:.]/g, "-");
        const filename = `batch_${String(batchIndex).padStart(4, "0")}_${timestamp}.json`;

        try {
          const fileHandle = await threadDirHandle.getFileHandle(filename, { create: true });
          const writable = await fileHandle.createWritable();
          const batchContent = {
            threadTitle,
            batchIndex,
            timestamp: now.toISOString(),
            messageCount: batchNewMessages.length,
            messages: batchNewMessages
          };

          await writable.write(JSON.stringify(batchContent, null, 2));
          await writable.close();
          
          newSavedMessagesCount += batchNewMessages.length;
          console.log(`Saved ${batchNewMessages.length} items to ${threadTitle}/${filename}`);
          batchIndex++;
        } catch (fileErr) {
          console.error("Error writing batch file:", fileErr);
        }
      }

      // Check if new DOM elements loaded after scrolling up
      if (messageNodes.length === previousNodeCount && batchNewMessages.length === 0) {
        unchangedCount++;
      } else {
        unchangedCount = 0;
      }
      previousNodeCount = messageNodes.length;

      // Force top scroll using multiple mechanisms
      forceScrollToTop(container);
      await sleep(2200); // Give Gemini time to fetch older history from backend
    }

    try {
      const summaryHandle = await threadDirHandle.getFileHandle("index.json", { create: true });
      const writable = await summaryHandle.createWritable();
      await writable.write(
        JSON.stringify({
          title: threadTitle,
          lastExtracted: new Date().toISOString(),
          totalUniqueMessagesSaved: existingHashes.size,
          lastBatchIndex: batchIndex - 1
        }, null, 2)
      );
      await writable.close();
    } catch (e) {
      console.warn("Failed writing index.json:", e);
    }

    return { totalMessages: existingHashes.size, newlyAdded: newSavedMessagesCount };
  };

  startBtn.addEventListener("click", async () => {
    if (!("showDirectoryPicker" in window)) {
      alert("Directory Picker API is not supported in this browser context.");
      return;
    }

    let rootDirHandle;
    try {
      rootDirHandle = await window.showDirectoryPicker({ mode: "readwrite" });
    } catch (err) {
      if (err.name === "AbortError") return;
      console.error("Failed to get directory handle:", err);
      return;
    }

    startBtn.innerText = "Extracting Threads...";
    startBtn.style.backgroundColor = "#d83b01";
    startBtn.disabled = true;

    const sidebarLinks = Array.from(
      document.querySelectorAll("a[href*='/app/'], conversation-item a, .conversation-link")
    );

    const threadTargets = sidebarLinks.map((link) => ({
      title: link.innerText.trim() || "Untitled_Thread",
      element: link,
      href: link.href
    })).filter((item, idx, self) => item.href && self.findIndex(t => t.href === item.href) === idx);

    console.log(`Discovered ${threadTargets.length} threads in sidebar.`);

    const globalManifest = {
      startedAt: new Date().toISOString(),
      totalThreadsFound: threadTargets.length,
      processedThreads: []
    };

    if (threadTargets.length === 0) {
      const currentTitle = document.title || "Current_Gemini_Thread";
      const threadFolder = await rootDirHandle.getDirectoryHandle(sanitizeFileName(currentTitle), { create: true });
      const stats = await processCurrentThread(threadFolder, currentTitle);
      globalManifest.processedThreads.push({ title: currentTitle, ...stats });
    } else {
      for (let i = 0; i < threadTargets.length; i++) {
        const target = threadTargets[i];
        startBtn.innerText = `Thread [${i + 1}/${threadTargets.length}]: ${target.title.slice(0, 15)}...`;

        target.element.click();
        await sleep(3500); // Allow Gemini to load thread contents

        const folderName = `${String(i + 1).padStart(3, "0")}_${sanitizeFileName(target.title)}`;
        const threadFolder = await rootDirHandle.getDirectoryHandle(folderName, { create: true });

        const stats = await processCurrentThread(threadFolder, target.title);
        globalManifest.processedThreads.push({ title: target.title, folder: folderName, ...stats });
      }
    }

    globalManifest.completedAt = new Date().toISOString();
    try {
      const manifestHandle = await rootDirHandle.getFileHandle("manifest.json", { create: true });
      const writable = await manifestHandle.createWritable();
      await writable.write(JSON.stringify(globalManifest, null, 2));
      await writable.close();
    } catch (e) {
      console.error("Failed writing manifest.json:", e);
    }

    startBtn.innerText = `Finished! Processed ${globalManifest.processedThreads.length} Threads`;
    startBtn.style.backgroundColor = "#107c41";
    setTimeout(() => startBtn.remove(), 5000);
  });
})();


Better gemini extractor that uses directories for threads, stores it in IDB, picks up where we left off, and scans the batchexecute request stream from the front-end to extract timestamps and sentiments

In [ ]:
(async function initGeminiEngine() {
  const DB_NAME = "GeminiExporterDB";
  const STORE_NAME = "handles";
  const HANDLE_KEY = "targetDirHandle";

  const sleep = (ms) => new Promise((resolve) => setTimeout(resolve, ms));

  // --- 1. IndexedDB Directory Handle Persistence ---
  const openDB = () => {
    return new Promise((resolve, reject) => {
      const request = indexedDB.open(DB_NAME, 1);
      request.onupgradeneeded = (e) => {
        const db = e.target.result;
        if (!db.objectStoreNames.contains(STORE_NAME)) {
          db.createObjectStore(STORE_NAME);
        }
      };
      request.onsuccess = () => resolve(request.result);
      request.onerror = () => reject(request.error);
    });
  };

  const saveDirectoryHandle = async (handle) => {
    const db = await openDB();
    return new Promise((resolve, reject) => {
      const tx = db.transaction(STORE_NAME, "readwrite");
      tx.objectStore(STORE_NAME).put(handle, HANDLE_KEY);
      tx.oncomplete = () => resolve();
      tx.onerror = () => reject(tx.error);
    });
  };

  const loadDirectoryHandle = async () => {
    try {
      const db = await openDB();
      return new Promise((resolve, reject) => {
        const tx = db.transaction(STORE_NAME, "readonly");
        const req = tx.objectStore(STORE_NAME).get(HANDLE_KEY);
        req.onsuccess = () => resolve(req.result || null);
        req.onerror = () => reject(req.error);
      });
    } catch {
      return null;
    }
  };

  // --- 2. String & Strict Timestamp Formatting ---
  const extractThreadIdFromUrl = (url = window.location.href) => {
    const match = url.match(/\/app\/([a-f0-9]{16})/i) || url.match(/\/app\/([a-zA-L0-9_-]+)/);
    return match ? match[1] : "default_thread";
  };

  const sanitizeFileName = (name) => {
    return (name || "Untitled")
      .replace(/[\\/:*?"<>|]/g, "_")
      .replace(/\s+/g, "_")
      .slice(0, 50);
  };

  const formatDateForPrefix = (dateObj) => {
    const d = dateObj || new Date();
    const year = d.getFullYear();
    const month = String(d.getMonth() + 1).padStart(2, "0");
    const day = String(d.getDate()).padStart(2, "0");
    return `${year}-${month}-${day}`;
  };

  const extractUnixTimestamp = (input) => {
    if (!input) return null;
    const str = typeof input === "string" ? input : JSON.stringify(input);

    const structuralMatch = str.match(/,\s*1\s*,\s*1\s*\]\s*,\s*\[\s*(\d{10,13})\s*,/);
    if (structuralMatch && structuralMatch[1]) {
      let epoch = parseInt(structuralMatch[1], 10);
      if (epoch < 1e11) epoch *= 1000;
      const date = new Date(epoch);
      if (!isNaN(date.getTime())) return date;
    }

    const arrayMatch = str.match(/\[\s*(1[6-9]\d{8})\s*,\s*\d+\s*\]/);
    if (arrayMatch && arrayMatch[1]) {
      const date = new Date(parseInt(arrayMatch[1], 10) * 1000);
      if (!isNaN(date.getTime())) return date;
    }

    const matches = str.match(/\b(1[6-9]\d{8})\b/g);
    if (matches) {
      const minEpochMs = new Date("2020-01-01T00:00:00Z").getTime();
      const maxEpochMs = new Date("2028-01-01T00:00:00Z").getTime();

      for (let i = matches.length - 1; i >= 0; i--) {
        const epochMs = parseInt(matches[i], 10) * 1000;
        if (epochMs >= minEpochMs && epochMs <= maxEpochMs) {
          return new Date(epochMs);
        }
      }
    }

    return null;
  };

  // --- 3. DOM Extractor ---
  const parseDOMMessages = (networkChunks = []) => {
    const nodes = Array.from(
      document.querySelectorAll(
        "user-query, model-response, .user-query, .model-response, [data-test-id='user-query']"
      )
    );

    return nodes.map((node, index) => {
      const isUser = node.tagName.toLowerCase().includes("user") || node.classList.contains("user-query");
      const role = isUser ? "user" : "model";
      const text = node.innerText ? node.innerText.trim() : "";

      const timestampEl = node.querySelector("time, .time-stamp, [class*='timestamp'], [class*='time']");
      const domTimestampRaw = timestampEl ? timestampEl.innerText.trim() : null;

      const codeBlocks = Array.from(node.querySelectorAll("pre, code")).map((c) => c.innerText.trim());
      const images = Array.from(node.querySelectorAll("img"))
        .map((img) => img.src || img.getAttribute("data-src"))
        .filter((src) => src && !src.startsWith("data:image/svg"));

      const sampleKeywords = text
        .slice(0, 120)
        .replace(/[^\w\s]/gi, "")
        .split(/\s+/)
        .filter((w) => w.length > 3);

      let resolvedMessageDate = extractUnixTimestamp(domTimestampRaw) || extractUnixTimestamp(text);

      if (!resolvedMessageDate && networkChunks.length > 0) {
        for (const chunk of networkChunks) {
          const chunkStr = JSON.stringify(chunk);
          if (sampleKeywords.some((kw) => chunkStr.includes(kw))) {
            const foundDate = extractUnixTimestamp(chunkStr);
            if (foundDate) {
              resolvedMessageDate = foundDate;
              break;
            }
          }
        }
      }

      return {
        index,
        role,
        text,
        domTimestamp: domTimestampRaw,
        messageTimestamp: resolvedMessageDate ? resolvedMessageDate.toISOString() : null,
        extractedAt: new Date().toISOString(),
        codeBlocks,
        images,
        keywords: sampleKeywords
      };
    });
  };

  // --- 4. Sidebar Controller & Link Filter ---
  const ensureSidebarOpen = async () => {
    // Direct target for the exact Gemini sidebar sparkle/toggle button
    const expandBtn = document.querySelector(
      'button[data-test-id="side-nav-sparkle-button"], button[aria-label="Open sidebar"]'
    );

    const sidebar = document.querySelector("mat-sidenav, nav, .side-nav, side-navigation-v2, [role='navigation']");

    // Checking offsetWidth / clientWidth to catch both collapsed state and zero-width
    const isCollapsed = !sidebar || sidebar.clientWidth < 150;

    if (expandBtn && isCollapsed) {
      expandBtn.click();
      await sleep(1000);
    }
  };

  const getSidebarConversationTargets = () => {
    const links = Array.from(document.querySelectorAll("a[href*='/app/']"));
    const validTargets = [];

    for (const link of links) {
      const href = link.href;
      const threadId = extractThreadIdFromUrl(href);

      if (!threadId || href.includes("/settings") || href.includes("/account") || link.closest("footer, [role='contentinfo']")) {
        continue;
      }

      const text = link.innerText.trim();
      if (!text || text.toLowerCase().includes("sign in") || text.toLowerCase().includes("manage account")) {
        continue;
      }

      validTargets.push({
        title: text.split("\n")[0] || "Untitled Thread",
        href,
        id: threadId
      });
    }

    return validTargets.filter((item, idx, self) => self.findIndex((t) => t.id === item.id) === idx);
  };

  const scrollSidebarToBottom = async () => {
    await ensureSidebarOpen();

    const findScrollableSidebar = () => {
      const selectors = [
        "infinite-scroller",
        "side-navigation-content",
        ".side-nav-history-container",
        "mat-sidenav-content",
        "mat-sidenav",
        "nav"
      ];

      for (const sel of selectors) {
        const els = Array.from(document.querySelectorAll(sel));
        const match = els.find((el) => {
          const style = window.getComputedStyle(el);
          const overflow = style.overflowY || style.overflow;
          return (overflow === "auto" || overflow === "scroll" || overflow === "overlay") && el.scrollHeight > el.clientHeight;
        });
        if (match) return match;
      }

      const sidebarHost = document.querySelector("mat-sidenav, nav, side-navigation-v2");
      if (sidebarHost) {
        const allDivs = Array.from(sidebarHost.querySelectorAll("div, section"));
        return allDivs.find((el) => {
          const style = window.getComputedStyle(el);
          return (style.overflowY === "auto" || style.overflowY === "scroll") && el.scrollHeight > el.clientHeight;
        }) || sidebarHost;
      }
      return null;
    };

    const container = findScrollableSidebar();
    if (!container) return;

    let previousLinkCount = 0;
    let unchangedCount = 0;

    while (unchangedCount < 3) {
      const links = container.querySelectorAll("a[href*='/app/']");
      if (links.length > 0) {
        links[links.length - 1].scrollIntoView({ behavior: "instant", block: "end" });
      }

      container.scrollTop = container.scrollHeight;
      container.dispatchEvent(new Event("scroll", { bubbles: true }));

      await sleep(1500);

      const currentLinkCount = container.querySelectorAll("a[href*='/app/']").length;

      if (currentLinkCount === previousLinkCount) {
        unchangedCount++;
      } else {
        unchangedCount = 0;
        previousLinkCount = currentLinkCount;
      }
    }
  };

  // --- 5. Dynamic Navigation helper ---
  const navigateToThread = async (targetId) => {
    let freshLink = document.querySelector(`a[href*='${targetId}']`);

    // If element is not in DOM (virtualized out), scroll it into view
    if (!freshLink) {
      const container = document.querySelector("mat-sidenav, nav, side-navigation-v2");
      if (container) container.scrollTop = 0;
      await sleep(500);
      freshLink = document.querySelector(`a[href*='${targetId}']`);
    }

    if (!freshLink) return false;

    freshLink.click();

    // Poll until URL updates AND DOM finishes rendering new messages
    let attempts = 0;
    while (attempts < 20) {
      await sleep(500);
      if (extractThreadIdFromUrl() === targetId) {
        await sleep(1500); // Allow Angular/React time to render the new conversation DOM
        return true;
      }
      attempts++;
    }
    return false;
  };

  // --- 6. Network Stream Parsing ---
  const parseBatchExecuteResponse = (rawText) => {
    const parsedPayloads = [];
    const lines = rawText.split("\n");

    const isMessagePayload = (data) => {
      if (!Array.isArray(data)) return false;
      let current = data;
      while (Array.isArray(current) && current.length > 0 && Array.isArray(current[0])) {
        const firstChild = current[0];
        if (
          firstChild.length >= 2 &&
          typeof firstChild[0] === "string" &&
          typeof firstChild[1] === "string" &&
          firstChild[0].startsWith("c_") &&
          (firstChild[1].startsWith("r_") || firstChild[1].startsWith("m_"))
        ) {
          return true;
        }
        current = firstChild;
      }
      return false;
    };

    for (const line of lines) {
      const trimmed = line.trim();
      if (!trimmed.startsWith("[") || !trimmed.endsWith("]")) continue;

      try {
        const outerArray = JSON.parse(trimmed);
        if (!Array.isArray(outerArray)) continue;

        for (const item of outerArray) {
          if (Array.isArray(item) && item[0] === "wrb.fr") {
            const rpcId = item[1];
            const innerPayloadStr = item[2];

            if (typeof innerPayloadStr === "string") {
              try {
                const nestedData = JSON.parse(innerPayloadStr);
                if (isMessagePayload(nestedData)) {
                  parsedPayloads.push({ rpcId, data: nestedData, rawTextSnippet: innerPayloadStr });
                }
              } catch {
                if (innerPayloadStr.includes('["c_') && (innerPayloadStr.includes('"r_') || innerPayloadStr.includes('"m_'))) {
                  parsedPayloads.push({ rpcId, rawTextSnippet: innerPayloadStr });
                }
              }
            }
          }
        }
      } catch { }
    }

    return parsedPayloads;
  };

  let activeDirHandle = await loadDirectoryHandle();
  let capturedNetworkChunks = [];

  // --- Intercept Fetch & XHR ---
  const originalFetch = window.fetch;
  window.fetch = async function (...args) {
    const response = await originalFetch.apply(this, args);
    const url = typeof args[0] === "string" ? args[0] : args[0]?.url || "";

    if (url.includes("batchexecute")) {
      try {
        const clone = response.clone();
        const text = await clone.text();
        const records = parseBatchExecuteResponse(text);
        if (records.length > 0) {
          capturedNetworkChunks.push(...records);
        }
      } catch (err) {
        console.error("Fetch intercept error:", err);
      }
    }
    return response;
  };

  const originalXhrOpen = XMLHttpRequest.prototype.open;
  const originalXhrSend = XMLHttpRequest.prototype.send;

  XMLHttpRequest.prototype.open = function (method, url, ...rest) {
    this._requestUrl = url;
    return originalXhrOpen.apply(this, [method, url, ...rest]);
  };

  XMLHttpRequest.prototype.send = function (...args) {
    if (this._requestUrl && this._requestUrl.includes("batchexecute")) {
      this.addEventListener("readystatechange", function () {
        if (this.readyState === 4 && this.status === 200) {
          try {
            const records = parseBatchExecuteResponse(this.responseText);
            if (records.length > 0) {
              capturedNetworkChunks.push(...records);
            }
          } catch (err) {
            console.error("XHR intercept error:", err);
          }
        }
      });
    }
    return originalXhrSend.apply(this, args);
  };

  // --- 7. UI Setup ---
  const existingPanel = document.getElementById("gemini-master-panel");
  if (existingPanel) existingPanel.remove();

  const panel = document.createElement("div");
  panel.id = "gemini-master-panel";

  const startBtn = document.createElement("button");
  startBtn.innerText = activeDirHandle ? "Start Ordered Export" : "Select Target Directory";

  const statusText = document.createElement("span");
  statusText.innerText = activeDirHandle ? ` Target: "${activeDirHandle.name}"` : " Ready";

  panel.appendChild(startBtn);
  panel.appendChild(statusText);
  document.body.appendChild(panel);

  const styleEl = document.createElement("style");
  styleEl.textContent = `
    #gemini-master-panel {
      position: fixed; bottom: 20px; right: 20px; z-index: 999999;
      display: flex; align-items: center; gap: 12px;
      background: #1e1e1e; border: 1px solid #333; padding: 10px 16px;
      border-radius: 8px; color: #fff; font-family: sans-serif;
      box-shadow: 0 4px 12px rgba(0,0,0,0.5);
    }
    #gemini-master-panel button {
      background: #1a73e8; color: white; border: none; padding: 8px 14px;
      border-radius: 6px; font-weight: bold; cursor: pointer;
    }
    #gemini-master-panel button:disabled { background: #555; }
  `;
  document.head.appendChild(styleEl);

  // --- 8. Thread Processing Logic ---
  const findMainScrollContainer = () => {
    const candidates = Array.from(document.querySelectorAll("main, .conversation-container, chat-app, div[class*='scroll']"));
    return candidates.find((el) => {
      const style = window.getComputedStyle(el);
      return (style.overflowY === "auto" || style.overflowY === "scroll") && el.scrollHeight > el.clientHeight;
    }) || document.documentElement;
  };

  const forceScrollToTop = (container) => {
    const firstMsg = document.querySelector("user-query, model-response, .user-query, .model-response");
    if (firstMsg) {
      firstMsg.scrollIntoView({ behavior: "instant", block: "start" });
    }
    if (container && container !== document.body && container !== document.documentElement) {
      container.scrollTop = 0;
      container.dispatchEvent(new Event("scroll", { bubbles: true }));
    }
    window.scrollTo({ top: 0, behavior: "instant" });
    window.dispatchEvent(new Event("scroll", { bubbles: true }));
  };

  const processThread = async (threadId, threadTitle) => {
    let currentTimestamp = extractUnixTimestamp(capturedNetworkChunks) || new Date();
    let datePrefix = formatDateForPrefix(currentTimestamp);
    let folderName = `${datePrefix}-${threadId}-${sanitizeFileName(threadTitle)}`;
    let threadFolderHandle = await activeDirHandle.getDirectoryHandle(folderName, { create: true });

    let batchIndex = 1;
    for await (const entry of threadFolderHandle.values()) {
      if (entry.kind === "file" && entry.name.startsWith("batch_")) {
        const m = entry.name.match(/batch_(\d+)_/);
        if (m) {
          const idx = parseInt(m[1], 10);
          if (idx >= batchIndex) batchIndex = idx + 1;
        }
      }
    }

    const container = findMainScrollContainer();
    let unchangedCount = 0;
    let previousMessageCount = 0;

    console.log(`Extracting [${folderName}] starting at Batch #${batchIndex}`);

    while (unchangedCount < 4) {
      await ensureSidebarOpen();
      forceScrollToTop(container);
      await sleep(2500);

      // Verify that the current URL matches the thread being processed
      if (extractThreadIdFromUrl() !== threadId) {
        console.warn(`URL mismatch during processing! Expected ${threadId}, found ${extractThreadIdFromUrl()}`);
        break;
      }

      const parsedDOMMessages = parseDOMMessages(capturedNetworkChunks);

      const rawDetectedDate = extractUnixTimestamp(capturedNetworkChunks) || extractUnixTimestamp(parsedDOMMessages);
      if (rawDetectedDate) {
        currentTimestamp = rawDetectedDate;
      }

      const messageTimestamp = currentTimestamp instanceof Date
        ? Math.floor(currentTimestamp.getTime() / 1000)
        : currentTimestamp;

      const isoDateStr = (currentTimestamp instanceof Date ? currentTimestamp : new Date(currentTimestamp * 1000))
        .toISOString()
        .replace(/[:.]/g, "-");

      const freshPrefix = formatDateForPrefix(currentTimestamp);
      if (freshPrefix !== datePrefix) {
        datePrefix = freshPrefix;
        folderName = `${datePrefix}-${threadId}-${sanitizeFileName(threadTitle)}`;
        threadFolderHandle = await activeDirHandle.getDirectoryHandle(folderName, { create: true });
      }

      if (parsedDOMMessages.length > 0) {
        const filename = `batch_${String(batchIndex).padStart(4, "0")}_${isoDateStr}.json`;

        try {
          const fileHandle = await threadFolderHandle.getFileHandle(filename, { create: true });
          const writable = await fileHandle.createWritable();

          const batchRecord = {
            threadId,
            threadTitle,
            folderName,
            batchIndex,
            messageTimestamp,
            currentUrl: window.location.href,
            extractedAt: new Date().toISOString(),
            threadDate: (currentTimestamp instanceof Date ? currentTimestamp : new Date(currentTimestamp * 1000)).toISOString(),
            totalDOMMessages: parsedDOMMessages.length,
            messages: parsedDOMMessages
          };

          await writable.write(JSON.stringify(batchRecord, null, 2));
          await writable.close();
          console.log(`Saved ${filename} for ${threadTitle}`);
          batchIndex++;
        } catch (err) {
          console.error("Batch write error:", err);
        }
      }

      if (parsedDOMMessages.length === previousMessageCount && capturedNetworkChunks.length === 0) {
        unchangedCount++;
      } else {
        unchangedCount = 0;
      }

      previousMessageCount = parsedDOMMessages.length;
      capturedNetworkChunks = [];
    }
  };

  // --- 9. Master Start Handler ---
  startBtn.addEventListener("click", async () => {
    try {
      if (!activeDirHandle) {
        activeDirHandle = await window.showDirectoryPicker({ mode: "readwrite" });
        await saveDirectoryHandle(activeDirHandle);
      } else {
        const query = await activeDirHandle.queryPermission({ mode: "readwrite" });
        if (query !== "granted") {
          await activeDirHandle.requestPermission({ mode: "readwrite" });
        }
      }

      startBtn.disabled = true;
      statusText.innerText = " Discovering all sidebar threads...";

      let previousTargetCount = 0;
      let targets = [];
      let staleRounds = 0;

      while (staleRounds < 2) {
        await scrollSidebarToBottom();
        targets = getSidebarConversationTargets();

        statusText.innerText = ` Discovered ${targets.length} thread(s)...`;

        if (targets.length === previousTargetCount) {
          staleRounds++;
        } else {
          staleRounds = 0;
          previousTargetCount = targets.length;
        }

        await sleep(1000);
      }

      console.log(`Final discovery complete. Total valid thread targets: ${targets.length}`);

      const activeId = extractThreadIdFromUrl();
      let startIndex = targets.findIndex((t) => t.id === activeId);

      if (startIndex !== -1) {
        statusText.innerText = ` Resuming from current thread: "${targets[startIndex].title.slice(0, 15)}..."`;
      } else {
        startIndex = 0;
      }

      for (let i = startIndex; i < targets.length; i++) {
        const target = targets[i];
        statusText.innerText = ` Thread [${i + 1}/${targets.length}]: ${target.title.slice(0, 15)}...`;

        if (extractThreadIdFromUrl() !== target.id) {
          capturedNetworkChunks = []; // Clear network buffer BEFORE navigating
          await ensureSidebarOpen();
          const navigated = await navigateToThread(target.id);
          if (!navigated) {
            console.warn(`Could not navigate to thread ${target.id}, skipping...`);
            continue;
          }
        }

        const currentId = extractThreadIdFromUrl();
        if (currentId === target.id) {
          await processThread(currentId, target.title);
        }
      }

      statusText.innerText = " Finished!";
      startBtn.innerText = "Completed";
      startBtn.disabled = false;
    } catch (err) {
      console.error("Extraction error:", err);
      statusText.innerText = " Failed";
      startBtn.disabled = false;
    }
  });
})();